Step 1: Dataset preparation

In [1]:

# Machine Learning Assignment 2
# Step 1: Dataset Preparation
# Dataset: ICRISAT District-Level Data

import pandas as pd

# 1. Load the dataset
df = pd.read_csv("ICRISAT-District Level Data.csv")

print("Original dataset shape:", df.shape)
print("\nFirst five rows:")
print(df.head())

# 2. Select the rice yield column
yield_column = "RICE YIELD (Kg per ha)"

# 3. Remove invalid rice-yield records
# -1 represents unavailable data; zero yield is excluded
df = df[df[yield_column] > 0].copy()

print("\nDataset shape after removing invalid rice yields:",
      df.shape)

# 4. Calculate the median rice yield
threshold = df[yield_column].median()

print("\nRice yield threshold:", round(threshold, 2))

# 5. Create the target variable
# 0 = Low Yield
# 1 = High Yield

df["Yield_Class"] = (
    df[yield_column] > threshold
).astype(int)

print("\nTarget class distribution:")
print(df["Yield_Class"].value_counts().sort_index())

# 6. Remove columns that could cause target leakage
# Rice yield defines the target.
# Rice area and production are directly related to rice yield.

columns_to_remove = [
    "RICE AREA (1000 ha)",
    "RICE PRODUCTION (1000 tons)",
    "RICE YIELD (Kg per ha)"
]

df = df.drop(columns=columns_to_remove)

# 7. Replace unavailable values (-1) with missing values
df = df.replace(-1, float("nan"))

# 8. Save the prepared dataset
output_file = "ICRISAT_Prepared_Data.csv"
df.to_csv(output_file, index=False)

# 9. Display final information
print("\nPrepared dataset shape:", df.shape)

print("\nNumber of input features:",
      df.shape[1] - 1)

print("\nMissing values in the prepared dataset:")
print(df.isnull().sum().sum())

print("\nFinal dataset preview:")
print(df.head())

print("\nDataset saved successfully as:", output_file)


Original dataset shape: (16146, 80)

First five rows:
   Dist Code  Year  State Code    State Name Dist Name  RICE AREA (1000 ha)  \
0          1  1966          14  Chhattisgarh      Durg                548.0   
1          1  1967          14  Chhattisgarh      Durg                547.0   
2          1  1968          14  Chhattisgarh      Durg                556.3   
3          1  1969          14  Chhattisgarh      Durg                563.4   
4          1  1970          14  Chhattisgarh      Durg                571.6   

   RICE PRODUCTION (1000 tons)  RICE YIELD (Kg per ha)  WHEAT AREA (1000 ha)  \
0                        185.0                  337.59                  44.0   
1                        409.0                  747.71                  50.0   
2                        468.0                  841.27                  53.7   
3                        400.8                  711.40                  49.4   
4                        473.6                  828.55                 

Step 2 — Preprocess data and split it

In [2]:

# Step 2: Data Preprocessing and Train-Test Split

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Load the prepared dataset
df = pd.read_csv("ICRISAT_Prepared_Data.csv")

# 2. Separate features and target
X = df.drop(columns=["Yield_Class"])
y = df["Yield_Class"]

# 3. Identify numerical and categorical columns
categorical_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_cols = X.select_dtypes(
    include=["number"]
).columns.tolist()

print("Numerical columns:", len(numerical_cols))
print("Categorical columns:", len(categorical_cols))

# 4. Preprocess numerical features
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 5. Preprocess categorical features
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# 6. Combine preprocessing steps
preprocessor = ColumnTransformer([
    ("numerical", numeric_pipeline, numerical_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

# 7. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 8. Fit preprocessing ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 9. Display results
print("\nTraining samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

print("\nProcessed training shape:",
      X_train_processed.shape)
print("Processed testing shape:",
      X_test_processed.shape)

print("\nTraining class distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting class distribution:")
print(y_test.value_counts().sort_index())

print("\nPreprocessing completed successfully.")


Numerical columns: 75
Categorical columns: 2

Training samples: 11984
Testing samples: 2997

Processed training shape: (11984, 400)
Processed testing shape: (2997, 400)

Training class distribution:
Yield_Class
0    5992
1    5992
Name: count, dtype: int64

Testing class distribution:
Yield_Class
0    1499
1    1498
Name: count, dtype: int64

Preprocessing completed successfully.


Step 3 — Train five ML classification models

In [3]:

# Step 3: Train and Evaluate Five Classification Models

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

# 1. Define the five classification models
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": SVC(
        probability=True,
        random_state=42
    )
}

# 2. Create a list to store evaluation results
results = []

# 3. Train and evaluate each model
for name, model in models.items():

    print("\n" + "=" * 60)
    print("Model:", name)

    # Train the model
    if name == "Naive Bayes":
        model.fit(
            X_train_processed.toarray()
            if hasattr(X_train_processed, "toarray")
            else X_train_processed,
            y_train
        )
        X_test_model = (
            X_test_processed.toarray()
            if hasattr(X_test_processed, "toarray")
            else X_test_processed
        )
    else:
        model.fit(X_train_processed, y_train)
        X_test_model = X_test_processed

    # Predict target classes
    y_pred = model.predict(X_test_model)

    # Predict probabilities for AUC
    y_prob = model.predict_proba(X_test_model)[:, 1]

    # Calculate all six metrics
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    precision = precision_score(
        y_test, y_pred, zero_division=0
    )
    recall = recall_score(
        y_test, y_pred, zero_division=0
    )
    f1 = f1_score(
        y_test, y_pred, zero_division=0
    )
    mcc = matthews_corrcoef(y_test, y_pred)

    # Store the results
    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "AUC": auc,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "MCC": mcc
    })

    # Display metrics
    print(f"Accuracy : {accuracy:.4f}")
    print(f"AUC      : {auc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"MCC      : {mcc:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(
        y_test, y_pred, zero_division=0
    ))

# 4. Display the comparison table
results_df = pd.DataFrame(results)

print("\n\nFINAL MODEL COMPARISON")
print(results_df.round(4).to_string(index=False))

# 5. Identify the model with the highest accuracy
best_model = results_df.loc[
    results_df["Accuracy"].idxmax()
]

print("\nBest model based on accuracy:",
      best_model["Model"])
print("Best accuracy:",
      round(best_model["Accuracy"], 4))

# 6. Save the comparison table
results_df.to_csv(
    "model_comparison_results.csv",
    index=False
)

print("\nResults saved to model_comparison_results.csv")



Model: Logistic Regression
Accuracy : 0.8969
AUC      : 0.9627
Precision: 0.8961
Recall   : 0.8979
F1 Score : 0.8970
MCC      : 0.7938

Confusion Matrix:
[[1343  156]
 [ 153 1345]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.90      0.90      1499
           1       0.90      0.90      0.90      1498

    accuracy                           0.90      2997
   macro avg       0.90      0.90      0.90      2997
weighted avg       0.90      0.90      0.90      2997


Model: Decision Tree
Accuracy : 0.8809
AUC      : 0.8809
Precision: 0.8816
Recall   : 0.8798
F1 Score : 0.8807
MCC      : 0.7618

Confusion Matrix:
[[1322  177]
 [ 180 1318]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.88      0.88      1499
           1       0.88      0.88      0.88      1498

    accuracy                           0.88      2997
   macro avg       0.88      0.88      0.88      

In [ ]:
Step 4 — Save all five trained models

In [4]:

# Step 4: Save Trained Models and Test Data

import os
import joblib
import pandas as pd
from sklearn.pipeline import Pipeline

# 1. Create project folders
os.makedirs("model", exist_ok=True)

# 2. Save all five trained models with preprocessing
for name, model in models.items():

    full_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    # The model was already trained on preprocessed data.
    # Save the fitted preprocessing and classifier together.
    model_filename = (
        name.lower().replace(" ", "_") + ".joblib"
    )

    joblib.dump(
        full_pipeline,
        os.path.join("model", model_filename)
    )

    print("Saved:", model_filename)

# 3. Create test data with the original feature columns
test_data = X_test.copy()
test_data["Yield_Class"] = y_test

test_data.to_csv("test_data.csv", index=False)

print("\nTest data saved successfully.")

# 4. Save model comparison results
results_df.to_csv(
    "model_comparison_results.csv",
    index=False
)

print("Model comparison results saved.")

# 5. Display saved files
print("\nFiles inside model folder:")
print(os.listdir("model"))

print("\nTest data shape:", test_data.shape)
print("\nStep 4 completed successfully.")


Saved: logistic_regression.joblib
Saved: decision_tree.joblib
Saved: knn.joblib
Saved: naive_bayes.joblib
Saved: support_vector_machine.joblib

Test data saved successfully.
Model comparison results saved.

Files inside model folder:
['decision_tree.joblib', 'knn.joblib', 'logistic_regression.joblib', 'naive_bayes.joblib', 'support_vector_machine.joblib']

Test data shape: (2997, 78)

Step 4 completed successfully.


Step 5 — Create requirements.txt

In [6]:

# Step 5: Create requirements.txt

requirements = """streamlit
pandas
numpy
scikit-learn
joblib
matplotlib
seaborn
"""

with open("requirements.txt", "w") as file:
    file.write(requirements)

print("requirements.txt created successfully!")

print("\nFile contents:")
print(requirements)


requirements.txt created successfully!

File contents:
streamlit
pandas
numpy
scikit-learn
joblib
matplotlib
seaborn



Step 6 — Create the Streamlit application (app.py).

In [5]:

# Step 6: Create the Streamlit Application

app_code = r'''
import streamlit as st
import pandas as pd
import joblib
import os

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

st.set_page_config(
    page_title="ICRISAT Yield Classification",
    page_icon="🌾",
    layout="wide"
)

st.title("🌾 Rice Yield Classification")
st.write(
    "Compare five machine learning classifiers "
    "using ICRISAT district-level agricultural data."
)

MODEL_DIR = "model"

model_files = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "K-Nearest Neighbors": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Support Vector Machine": "support_vector_machine.joblib"
}

st.header("1. Upload Test Dataset")

uploaded_file = st.file_uploader(
    "Upload a CSV file containing input features and Yield_Class",
    type=["csv"]
)

if uploaded_file is not None:
    try:
        test_data = pd.read_csv(uploaded_file)

        st.success("CSV file uploaded successfully.")
        st.write("Dataset shape:", test_data.shape)
        st.dataframe(test_data.head())

        if "Yield_Class" not in test_data.columns:
            st.error(
                "The uploaded CSV must contain the Yield_Class target column."
            )
            st.stop()

        y_test = test_data["Yield_Class"].astype(int)
        X_test = test_data.drop(columns=["Yield_Class"])

        if not y_test.isin([0, 1]).all():
            st.error("Yield_Class must contain only 0 and 1.")
            st.stop()

        st.header("2. Select Classification Model")

        selected_model = st.selectbox(
            "Choose a model",
            list(model_files.keys())
        )

        model_path = os.path.join(
            MODEL_DIR,
            model_files[selected_model]
        )

        if not os.path.exists(model_path):
            st.error("Saved model file not found: " + model_path)
            st.stop()

        if st.button("Evaluate Model"):
            try:
                pipeline = joblib.load(model_path)

                # Predict directly from the original CSV columns
                y_pred = pipeline.predict(X_test)

                # Get positive-class probabilities for AUC
                y_prob = pipeline.predict_proba(X_test)[:, 1]

                accuracy = accuracy_score(y_test, y_pred)
                auc = roc_auc_score(y_test, y_prob)
                precision = precision_score(
                    y_test, y_pred, zero_division=0
                )
                recall = recall_score(
                    y_test, y_pred, zero_division=0
                )
                f1 = f1_score(
                    y_test, y_pred, zero_division=0
                )
                mcc = matthews_corrcoef(y_test, y_pred)

                st.header("3. Evaluation Metrics")

                col1, col2, col3 = st.columns(3)
                col1.metric("Accuracy", f"{accuracy:.4f}")
                col2.metric("AUC Score", f"{auc:.4f}")
                col3.metric("Precision", f"{precision:.4f}")

                col4, col5, col6 = st.columns(3)
                col4.metric("Recall", f"{recall:.4f}")
                col5.metric("F1 Score", f"{f1:.4f}")
                col6.metric("MCC Score", f"{mcc:.4f}")

                st.header("4. Confusion Matrix")

                cm = confusion_matrix(
                    y_test, y_pred, labels=[0, 1]
                )

                st.write(
                    pd.DataFrame(
                        cm,
                        index=["Actual Low Yield (0)",
                               "Actual High Yield (1)"],
                        columns=["Predicted Low Yield (0)",
                                 "Predicted High Yield (1)"]
                    )
                )

                st.header("5. Classification Report")

                report = classification_report(
                    y_test,
                    y_pred,
                    target_names=["Low Yield", "High Yield"],
                    output_dict=True,
                    zero_division=0
                )

                st.dataframe(
                    pd.DataFrame(report).transpose().round(4)
                )

            except Exception as error:
                st.error(
                    "Model evaluation failed. Check the saved "
                    "pipeline and uploaded CSV columns."
                )
                st.exception(error)

    except Exception as error:
        st.error("Could not read the uploaded CSV file.")
        st.exception(error)

else:
    st.info("Please upload your test_data.csv file to begin.")

st.caption(
    "Machine Learning Assignment 2 | ICRISAT Dataset"
)
'''

with open("app.py", "w", encoding="utf-8") as file:
    file.write(app_code)

print("app.py created successfully!")
print("File size:", os.path.getsize("app.py"), "bytes")


app.py created successfully!
File size: 4987 bytes


Step 7 — Test the saved models

In [6]:

# Step 7: Verify all saved model pipelines

import os
import joblib
import pandas as pd

from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# Load the test dataset
test_data = pd.read_csv("test_data.csv")

X_verify = test_data.drop(columns=["Yield_Class"])
y_verify = test_data["Yield_Class"]

# Model file names
model_files = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Support Vector Machine": "support_vector_machine.joblib"
}

print("Testing saved models...\n")

for name, filename in model_files.items():

    path = os.path.join("model", filename)

    try:
        saved_pipeline = joblib.load(path)

        # GaussianNB requires dense input.
        if name == "Naive Bayes":
            saved_pipeline.steps.insert(
                1,
                (
                    "to_dense",
                    FunctionTransformer(
                        lambda x: x.toarray()
                        if hasattr(x, "toarray") else x
                    )
                )
            )

        predictions = saved_pipeline.predict(X_verify)
        accuracy = accuracy_score(y_verify, predictions)

        print(name, "- PASSED")
        print("Accuracy:", round(accuracy, 4))
        print("Predictions generated:", len(predictions))
        print("-" * 40)

        # Save the corrected pipeline
        joblib.dump(saved_pipeline, path)

    except Exception as error:
        print(name, "- FAILED")
        print("Error:", error)
        print("-" * 40)

print("\nModel verification completed.")


Testing saved models...

Logistic Regression - PASSED
Accuracy: 0.8969
Predictions generated: 2997
----------------------------------------
Decision Tree - PASSED
Accuracy: 0.8809
Predictions generated: 2997
----------------------------------------
KNN - PASSED
Accuracy: 0.8976
Predictions generated: 2997
----------------------------------------
Naive Bayes - PASSED
Accuracy: 0.8001
Predictions generated: 2997
----------------------------------------
Naive Bayes - FAILED
Error: Can't pickle <function <lambda> at 0x0000020AD2294400>: it's not found as __main__.<lambda>
----------------------------------------
Support Vector Machine - PASSED
Accuracy: 0.9009
Predictions generated: 2997
----------------------------------------

Model verification completed.


Step 7A — Correct the Naive Bayes model

In [9]:

# Step 7A: Fix and Save the Naive Bayes Pipeline

import os
import joblib
from sklearn.pipeline import Pipeline

# 1. Create a separate module for dense conversion
transformer_code = '''
class DenseTransformer:
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if hasattr(X, "toarray"):
            return X.toarray()
        return X
'''

with open("dense_transformer.py", "w", encoding="utf-8") as file:
    file.write(transformer_code)

# 2. Import the transformer from the module
import importlib
import dense_transformer
importlib.reload(dense_transformer)

DenseTransformer = dense_transformer.DenseTransformer

# 3. Build the corrected pipeline using fitted components
nb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("to_dense", DenseTransformer()),
    ("classifier", models["Naive Bayes"])
])

# 4. Save the corrected pipeline
joblib.dump(
    nb_pipeline,
    "model/naive_bayes.joblib"
)

print("Corrected Naive Bayes pipeline saved.")

# 5. Verify the saved pipeline
loaded_nb = joblib.load("model/naive_bayes.joblib")

X_check = X_test.iloc[:10]
predictions = loaded_nb.predict(X_check)

print("Verification successful!")
print("Number of predictions:", len(predictions))
print("Predictions:", predictions)

print("\nRemember to include dense_transformer.py in GitHub.")


Corrected Naive Bayes pipeline saved.
Verification successful!
Number of predictions: 10
Predictions: [1 1 0 1 0 1 0 1 0 0]

Remember to include dense_transformer.py in GitHub.


In [ ]:
Next step — Step 8: Create README.md

In [7]:

# Step 8: Create README.md

readme = """# Rice Yield Classification Using Machine Learning

## 1. Problem Statement

The objective of this project is to classify agricultural records
into High Yield and Low Yield categories using ICRISAT district-level
agricultural data. Five classification algorithms are implemented
and compared using standard evaluation metrics.

## 2. Dataset Description

- Dataset: ICRISAT District-Level Data
- Original records: 16,146
- Original columns: 80
- Records retained after yield filtering: 14,981
- Target column: Yield_Class
- Class 0: Low Yield
- Class 1: High Yield
- Yield threshold: 1,417.1 kg/ha
- Training records: 11,984
- Testing records: 2,997

Records with non-positive rice yield were excluded. Rice area,
rice production, and rice yield were removed from the input features
to prevent target leakage.

## 3. GitHub Repository Link

Add your GitHub repository URL here after creating the repository.

## 4. Models Used and Evaluation Results

The following results were obtained from the test dataset.

| ML Model | Accuracy | AUC | Precision | Recall | F1 Score | MCC |
|---|---:|---:|---:|---:|---:|---:|
| Logistic Regression | 0.8969 | 0.9627 | 0.8961 | 0.8979 | 0.8970 | 0.7938 |
| Decision Tree | 0.8809 | 0.8809 | 0.8816 | 0.8798 | 0.8807 | 0.7618 |
| KNN | 0.8976 | 0.9528 | 0.8994 | 0.8952 | 0.8973 | 0.7951 |
| Naive Bayes | 0.8001 | 0.8310 | 0.7684 | 0.8591 | 0.8112 | 0.6045 |
| Support Vector Machine | 0.9009 | 0.9644 | 0.8979 | 0.9045 | 0.9012 | 0.8018 |

## 5. Observations

### Logistic Regression
Achieved 89.69% accuracy and an AUC score of 0.9627,
indicating strong classification performance.

### Decision Tree
Achieved 88.09% accuracy. Its performance was lower than
Logistic Regression, KNN, and SVM in this experiment.

### K-Nearest Neighbors
Achieved 89.76% accuracy and 0.8994 precision.
It performed competitively with Logistic Regression.

### Naive Bayes
Achieved 80.01% accuracy and 0.8310 AUC.
It had the lowest accuracy among the five models, but its
recall was 0.8591.

### Support Vector Machine
Achieved the highest accuracy (90.09%), AUC (0.9644),
F1 score (0.9012), and MCC (0.8018) among the five models.

### Overall Winner
Support Vector Machine was the best-performing model in
this experiment based on the reported test metrics.

## 6. Project Files

- app.py: Streamlit application
- requirements.txt: Python dependencies
- test_data.csv: Test dataset with target column
- model/: Saved classification pipelines
- dense_transformer.py: Dense conversion transformer
- model_comparison_results.csv: Model evaluation results

## 7. How to Run

Install dependencies:

    pip install -r requirements.txt

Run the Streamlit application:

    streamlit run app.py

Upload a test CSV containing the required input features and
Yield_Class target column. Select a model to view the evaluation
metrics, confusion matrix, and classification report.
"""

with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme)

print("README.md created successfully!")
print("README file size:", len(readme), "characters")


README.md created successfully!
README file size: 2892 characters


Step 9A — Check the required project files

In [8]:

# Step 9A: Verify Project Files

import os

required_files = [
    "app.py",
    "requirements.txt",
    "README.md",
    "test_data.csv",
    "model_comparison_results.csv",
    "dense_transformer.py",
    "model/logistic_regression.joblib",
    "model/decision_tree.joblib",
    "model/knn.joblib",
    "model/naive_bayes.joblib",
    "model/support_vector_machine.joblib"
]

print("PROJECT FILE VERIFICATION")
print("-" * 45)

all_found = True

for filename in required_files:
    if os.path.isfile(filename):
        print("[OK]     ", filename)
    else:
        print("[MISSING]", filename)
        all_found = False

print("-" * 45)

if all_found:
    print("All required project files are available.")
else:
    print("Please create or restore the missing files.")


PROJECT FILE VERIFICATION
---------------------------------------------
[OK]      app.py
[OK]      requirements.txt
[OK]      README.md
[OK]      test_data.csv
[OK]      model_comparison_results.csv
[OK]      dense_transformer.py
[OK]      model/logistic_regression.joblib
[OK]      model/decision_tree.joblib
[OK]      model/knn.joblib
[OK]      model/naive_bayes.joblib
[OK]      model/support_vector_machine.joblib
---------------------------------------------
All required project files are available.


In [ ]:
!streamlit run app.py

In [9]:

import pandas as pd
import joblib
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC

print("Scikit-learn version:", sklearn.__version__)

# Load prepared dataset
data = pd.read_csv("ICRISAT_Prepared_Data.csv")

# Separate features and target
X = data.drop(columns=["Yield_Class"])
y = data["Yield_Class"].astype(int)

# Identify numerical and categorical columns
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

# Numerical preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Build SVM pipeline
svm_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", SVC(
        probability=True,
        random_state=42
    ))
])

# Train SVM
svm_pipeline.fit(X_train, y_train)

# Test predictions
y_pred = svm_pipeline.predict(X_test)

print("SVM training completed successfully.")
print("Test accuracy:", sklearn.metrics.accuracy_score(y_test, y_pred))

# Save the new SVM model
joblib.dump(
    svm_pipeline,
    "support_vector_machine.joblib"
)

print("New SVM model saved successfully.")


Scikit-learn version: 1.7.2
SVM training completed successfully.
Test accuracy: 0.9009009009009009
New SVM model saved successfully.


In [10]:

import os
import pandas as pd
import joblib
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("Scikit-learn version:", sklearn.__version__)

# Load prepared dataset
data = pd.read_csv("ICRISAT_Prepared_Data.csv")

# Separate features and target
X = data.drop(columns=["Yield_Class"])
y = data["Yield_Class"].astype(int)

# Identify numerical and categorical columns
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

# Numerical preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Build Logistic Regression pipeline
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000))
])

# Train model
logistic_pipeline.fit(X_train, y_train)

# Test model
y_pred = logistic_pipeline.predict(X_test)

print("Training completed successfully.")
print("Test accuracy:", accuracy_score(y_test, y_pred))

# Save model
os.makedirs("model", exist_ok=True)

joblib.dump(
    logistic_pipeline,
    "model/logistic_regression.joblib"
)

print("New Logistic Regression model saved successfully.")


Scikit-learn version: 1.7.2
Training completed successfully.
Test accuracy: 0.8968968968968969
New Logistic Regression model saved successfully.
